In [ ]:
# TITLE
# RadarNetCDFloader.ipynb
# TITLE

import numpy as np
from matplotlib import pyplot as plt
import xarray as xr

# import zipfile as zp          # used for unzipping ppi files
from pathlib import Path      # used to play with pathnames to save 
# from datetime import datetime # used to manipulate time :)

# import wradlib as wr          # used for having fun with radar data

from PIL import Image         # used for creating gif loops
import os                     # used for retrieving file names

# import h5py                   # used for reading .h5 files (Radar Level 1 data)
# import h5netcdf               # used for converting .h5 files to NetCDF

# # TAKEN FROM "Part4IntroductionToGridding"
# import cartopy.crs as ccrs
# import pyart
# import cartopy.crs as ccrs
# import cartopy.feature as cfeature
# import matplotlib.ticker as mticker

# mapping things
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader

# for adding lat/lon gridlines on plots
import matplotlib.ticker as mticker
from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER

# # SPECIAL METHOD TO IMPORT LEROI RADAR GRIDDING PACKAGE AND CUSTOM FUNCTIONS FROM LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks/CustomFunctions')
from CustomFunctions1 import *
# from leroi.leroi import *

# for adding a colourful topo base map to the CAPI plots
from custom_elevation import fetch_srtm, fetch_gebco_local
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import ListedColormap, BoundaryNorm

In [ ]:
# Load in a DEM (update path and variable name as needed)
DEMpath = '/home/563/sg3241/QueenslandElevationGEBCO.nc'  # <- your DEM file
DEMdata = xr.open_dataset(DEMpath)
DEMelev = DEMdata['elevation']  # adjust if your var has a different name

In [ ]:
# VERTICAL CROSS SECTION PLOTTING
# (LOADS IN FROM NET CDF FILES STORED IN SCRATCH)

# CHOOSE THE RADAR
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Bris)

# CHOOSE THE DATE
# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 2
RadarDay   = 14

# CHOOSE THE SLICE OF THE CROSS-SECTION
EWsliceKM = 40 # [km]     # number of km north or south of the radar you want to take the east-west slice for x-section
EWsliceNorS = 'South'    # direction ['North' or 'South'] from the radar you want the slice taken

# CHOOSE YOUR ALTITUDE RANGE
MinHeight = 0 # [km] minimum height in plot
MaxHeight = 15 # [km] maximum height in the plot

# PLEASE MAKE IT SO THE PLOT VARIABLES CAN BE CHOSEN HERE!!!! (ADD STRING EXECUTERS AND SUCH)
# CHOOSE THE VARIABLE TO PLOT
Var  = 'CC'

# LIST OF POSSIBLE VARIABLES
# [Z]         'corrected_reflectivity'
# [CC]        'corrected_cross_correlation_ratio'
# [ZDR]       'corrected_differential_reflectivity'
# [KDP]       'corrected_specific_differential_phase'
# [PhiDP]     'corrected_differential_phase'
# [IntAtt]    'path_integrated_attenuation'
# [DifIntAtt] 'path_integrated_differential_attenuation'
# [EchClas]   'radar_echo_classification'
# [V]         'corrected_velocity'
# [AzSh]      'azshear'



# USER CHOICE FOLLOW-ON SECTION

# radar choice follow-on
if (RadarIDno == '22'):
    RadarSiteName = 'Mackay'
elif (RadarIDno == '106'):
    RadarSiteName = 'Townsville'
elif (RadarIDno == '66'):
    RadarSiteName = 'Mt Staplyton'
elif (RadarIDno == '50'):
    RadarSiteName = 'Marabong'
else:
    RadarSiteName = 'Site ' + RadarIDno


# date choice follow-on
# add leading zeros for strings
YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)
# write out the data in one string with and without dashes
RadarFileDate  = YYYY + MM + DD
RadarFileDatePrint = YYYY + '-' + MM + '-' + DD


# slice choice follow-on
# THIS CODE ASSUMES THE X AND Y GRIDS ARE EVERY 1 KM
# IT WILL HAVE TO BE REWRITTEN TO CONSIDER IN-BETWEEN VALUES
# THIS CODE ASSUMES THE X AND Y GRIDS ARE EVERY 1 KM
# IT WILL HAVE TO BE REWRITTEN TO CONSIDER IN-BETWEEN VALUES
Slice    = str(EWsliceKM) + 'kmEW'
# add a positive or negative sign to the slice for math
if (EWsliceNorS == 'South'):
    EWsliceKMsign = EWsliceKM * -1
elif (EWsliceNorS == 'North'):
    EWsliceKMsign = EWsliceKM * 1
else:
    print("Please choose EWsliceNorS to be 'North' or 'South'.")


# variable choice follow-on
if (Var == 'Z'):
    VarName     = 'Reflectivity'
    VarNameLong = 'corrected_reflectivity'
    VarMinVal = -10 # [dBZ]
    VarMaxVal =  60 # [dBZ]
    VarUnit   = 'dBZ'
    VarColourBar = 'nipy_spectral'
elif (Var == 'ZDR'):
    VarName     = 'Differential Reflectivity'
    VarNameLong = 'corrected_differential_reflectivity'
    VarMinVal = -5 # [dB]
    VarMaxVal =  5 # [dB]
    VarUnit   = 'dB'
    VarColourBar = 'RdBu'
elif (Var == 'CC'):
    VarName     = 'Correlation Coefficient'
    VarNameLong = 'corrected_cross_correlation_ratio'
    VarMinVal = 0.8 # [0 to 1]
    VarMaxVal = 1.0 # [0 to 1]
    VarUnit   = '-0 to 1'
    VarColourBar = 'nipy_spectral'
elif (Var == 'KDP'):
    VarName     = 'Specific Differential Phase'
    VarNameLong = 'corrected_specific_differential_phase'
    VarMinVal = 0  # [deg/ km]
    VarMaxVal = 10 # [deg / km]
    VarUnit   = 'deg / km'
    VarColourBar = 'nipy_spectral'
elif (Var == 'PhiDP'):
    VarName     = 'Differential Phase'
    VarNameLong = 'corrected_differential_phase'
    VarMinVal = 0  # [deg]
    VarMaxVal = 30 # [deg]
    VarUnit   = 'deg'
    VarColourBar = 'nipy_spectral'
elif (Var == 'V'):
    VarName     = 'Velocity'
    VarNameLong = 'corrected_velocity'
    VarMinVal = -30 # [m/s]
    VarMaxVal =  30 # [m/s]
    VarUnit   = 'm/s'
    VarColourBar = 'RdBu_r'
else:
    raise ValueError("Input Variable '" + Var + "' not available\n" + "Please choose from the following list:\n" + \
          "[Z] 'corrected_reflectivity', [CC] 'corrected_cross_correlation_ratio', [ZDR] 'corrected_differential_reflectivity'\n" + \
          "[KDP] 'corrected_specific_differential_phase', [PhiDP] 'corrected_differential_phase'")

    

# remember these plots are vertical cross sections
PlotType = 'Vert'
# LOOP OVER EVERY 5 MIN PERIOD IN THE DAY
for houri in range(12,13):
    for mini in range(55,60,5):
        
        RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
        RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] 
        
        # add a string of format hh:mm:ss for printing
        print('working on ' + RadarFileTimePrint)

        NetCDFstoragePath = ('/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/' + \
                                RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc')
        # try to load in the netcdf file and if it doesn't work, just keep going through the loop
        try:
            xgrid = xr.open_dataset(NetCDFstoragePath)
        except FileNotFoundError:
            print(f'File missing for {RadarFileTimePrint}, skipping: {NetCDFstoragePath}')
            continue
        
        # THIS CODE ASSUMES THE X AND Y GRIDS ARE EVERY 1 KM
        # IT WILL HAVE TO BE REWRITTEN TO CONSIDER IN-BETWEEN VALUES

        # Coordinates in km (range from radar)
        yvalsKM = np.array(xgrid.x) * 0.001  # x coordinates in km
        # farthest south and north y-values
        MinSliceKM = int(np.min(yvalsKM))
        MaxSliceKM = int(np.max(yvalsKM))

        # find the vertical cross section index in the coordinates
        EWslicei = np.where(yvalsKM == EWsliceKMsign)[0][0]  # index in the x-coordinates where that north-south km value lives

        # EXPERIMENTAL TOPOGRAPHY SECTION
        # EXPERIMENTAL TOPOGRAPHY SECTION
        # EXPERIMENTAL TOPOGRAPHY SECTION

        # lat/lon fields in xgrid: dimensions (y, x)
        # Slice at the chosen north–south index EWslicei along x
        lat_slice = xgrid['lat'][EWslicei, :].values  # shape (nx,)
        lon_slice = xgrid['lon'][EWslicei, :].values  # shape (nx,)

        # Build DataArrays for interpolation
        LONdata = xr.DataArray(lon_slice, dims=('x',))
        LATdata = xr.DataArray(lat_slice, dims=('x',))

        # Interpolate DEM to the cross-section line
        dem_slice = DEMelev.interp(lon=LONdata, lat=LATdata)

        # Elevation in metres along the line
        TerrainSlice = dem_slice.values

        # For plotting in km, and do not let negative (ocean) go below 0
        TerrainSliceKM = np.maximum(TerrainSlice, 0.0) * 0.001

        Xkm = xgrid.x * 0.001  # east–west distance [km]

        # EXPERIMENTAL TOPOGRAPHY SECTION END
        # EXPERIMENTAL TOPOGRAPHY SECTION END
        # EXPERIMENTAL TOPOGRAPHY SECTION END
            
        fig, ax = plt.subplots(figsize=(8,6))
        GridViewer = pcolormeshC(lon_slice, xgrid.z*0.001, xgrid[VarNameLong][0,:,EWslicei,:], ax=ax, cmap=VarColourBar, vmin=VarMinVal, vmax=VarMaxVal)
        # mutiply by 0.001 to get distances in km

        # Plot the line showing ground/topography
        ax.plot(lon_slice, TerrainSliceKM, color=[0.3,0.3,0.3], linewidth=1.5, zorder=5 )
        ax.fill_between(lon_slice, 0, TerrainSliceKM , color=[0.3,0.3,0.3], zorder=4 ) # Shade everything below the terrain line in black


        # approximate latitude where the slice is taken (may vary a bit since the radar grid isn't perfect)
        ApproxLat = np.round(lat_slice[150], 2)

        # change it to positive and write 'north' or 'south'
        if (ApproxLat < 0):
            ApproxLat = ApproxLat * -1
            ApproxLatDir = 'S'
        else:
            ApproxLatDir = 'N'
        
        ax.set_xlabel('Longitude [Degrees East]')
        ax.set_ylabel('Altitude [km]')
        plt.title(VarName + ' Cross Section for ' + RadarSiteName + ' Radar\n For Slice Taken ' + \
              str(EWsliceKM) + ' km ' + EWsliceNorS + ' of the Radar (~' + str(ApproxLat) + '° ' + ApproxLatDir + ')\n' + \
              RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + ' at ' + \
              str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] + ' UTC')
        plt.colorbar(GridViewer, ax=ax, label = 'Reflectivity [dBZ]')
        plt.grid()
        
        plt.xlim([np.min(lon_slice), np.max(lon_slice)])
        plt.ylim([0,MaxHeight])
        
        SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/' + PlotType + '/' + RadarIDno + '/' + RadarFileDate + '/'
        SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + VarNameLong + '_' + PlotType + \
        str(EWsliceKM) + EWsliceNorS + '.png'
        
        SavePath = SaveFolder + SaveFile
        
        if not Path(SaveFolder).exists():
            print('Creating Folder: ' + SaveFolder)
            Path(SaveFolder).mkdir(parents=True, exist_ok=True)
        
        # plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)
        # plt.close()

In [ ]:
# GIF MAKER
# FOR Vertical Cross Sections

PlotVar  = 'corrected_reflectivity'

SavedFolder = '/scratch/v46/sg3241/tmp/pngImages/Vert/' + RadarIDno + '/' + RadarFileDate + '/'
                
# LOADING IMAGES
files = sorted(os.listdir(SavedFolder)) # takes all of the files in the folder in the order they are named
images = [
    Image.open(os.path.join(SavedFolder, f))
    for f in files
    if f.endswith(str(EWsliceKM) + EWsliceNorS + '.png')
]

# SaveFolder = '/cratch/v46/sg3241/tmp/pngImages/Vert/22/20240214/'

#         SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/CFAD/' + RadarIDno + '/' + RadarFileDate + '/'
#         SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + PlotVar + '_' + Version + 'CFAD.png'

GIFsaveFolder = '/scratch/v46/sg3241/tmp/gifImages/Vert/' + RadarIDno + '/' + RadarFileDate + '/'
GIFsaveFile   = RadarIDno + '_' + RadarFileDate + '_' + PlotVar + '_' + str(EWsliceKM) + EWsliceNorS + '.gif'

GIFsavePath = GIFsaveFolder + GIFsaveFile

if not Path(GIFsaveFolder).exists():
    Path(GIFsaveFolder).mkdir(parents=True, exist_ok=True)

# Save as looping GIF
images[0].save(
    GIFsavePath,
    save_all=True,
    append_images=images[1:],
    duration=200,    # ms per frame
    loop=0,          # 0 = loop forever
)
print('Saved GIF for ' + RadarFileDate)

In [ ]:
# HORIZONTAL CROSS SECTION PLOTTING (LOADS IN FROM NET CDF FILES STORED IN SCRATCH)

# VERTICAL CROSS SECTION PLOTTING
# (LOADS IN FROM NET CDF FILES STORED IN SCRATCH)

# CHOOSE THE RADAR
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Brisbane)

# CHOOSE THE DATE
# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 2
RadarDay   = 14

# CHOOSE YOUR ALTITUDE
Altitude = 10000  # [m] choose a multiple of 500 m to look at a CAPI for

# CHOOSE YOUR VARIABLE
Var = 'Z'
# LIST OF POSSIBLE VARIABLES
# [Z]         'corrected_reflectivity'
# [CC]        'corrected_cross_correlation_ratio'
# [ZDR]       'corrected_differential_reflectivity'
# [KDP]       'corrected_specific_differential_phase'
# [PhiDP]     'corrected_differential_phase'
...
# [V]         'corrected_velocity'

# VARIABLES NOT YET INCLUDED IN THE CODE
# [IntAtt]    'path_integrated_attenuation'
# [DifIntAtt] 'path_integrated_differential_attenuation'
# [EchClas]   'radar_echo_classification'
...
# [AzSh]      'azshear'



# USER CHOICE FOLLOW-ON SECTION

# ra
if (RadarIDno == '22'):
    RadarSiteName = 'Mackay'
elif (RadarIDno == '106'):
    RadarSiteName = 'Townsville'
elif (RadarIDno == '66'):
    RadarSiteName = 'Mt Staplyton'
elif (RadarIDno == '50'):
    RadarSiteName = 'Marabong'
else:
    RadarSiteName = 'Site ' + RadarIDno


# date choice follow-on
# add leading zeros for strings
YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)
# write out the data in one string with and without dashes
RadarFileDate  = YYYY + MM + DD
RadarFileDatePrint = YYYY + '-' + MM + '-' + DD


# variable choice follow-on
if (Var == 'Z'):
    VarName     = 'Reflectivity'
    VarNameLong = 'corrected_reflectivity'
    VarMinVal = -10 # [dBZ]
    VarMaxVal =  60 # [dBZ]
    VarUnit   = 'dBZ'
    VarColourBar = 'nipy_spectral'
elif (Var == 'ZDR'):
    VarName     = 'Differential Reflectivity'
    VarNameLong = 'corrected_differential_reflectivity'
    VarMinVal = -5 # [dB]
    VarMaxVal =  5 # [dB]
    VarUnit   = 'dB'
    VarColourBar = 'RdBu'
elif (Var == 'CC'):
    VarName     = 'Correlation Coefficient'
    VarNameLong = 'corrected_cross_correlation_ratio'
    VarMinVal = 0.8 # [0 to 1]
    VarMaxVal = 1.0 # [0 to 1]
    VarUnit   = '-0 to 1'
    VarColourBar = 'nipy_spectral'
elif (Var == 'KDP'):
    VarName     = 'Specific Differential Phase'
    VarNameLong = 'corrected_specific_differential_phase'
    VarMinVal = 0  # [deg/ km]
    VarMaxVal = 10 # [deg / km]
    VarUnit   = 'deg / km'
    VarColourBar = 'nipy_spectral'
elif (Var == 'PhiDP'):
    VarName     = 'Differential Phase'
    VarNameLong = 'corrected_differential_phase'
    VarMinVal = 0  # [deg]
    VarMaxVal = 30 # [deg]
    VarUnit   = 'deg'
    VarColourBar = 'nipy_spectral'
elif (Var == 'V'):
    VarName     = 'Velocity'
    VarNameLong = 'corrected_velocity'
    VarMinVal = -30 # [m/s]
    VarMaxVal =  30 # [m/s]
    VarUnit   = 'm/s'
    VarColourBar = 'RdBu_r'
else:
    raise ValueError("Input Variable '" + Var + "' not available\n" + "Please choose from the following list:\n" + \
          "[Z] 'corrected_reflectivity', [CC] 'corrected_cross_correlation_ratio', [ZDR] 'corrected_differential_reflectivity'\n" + \
          "[KDP] 'corrected_specific_differential_phase', [PhiDP] 'corrected_differential_phase'")

    

# remember these plots are horizontal cross sections
PlotType = 'Horz'
# LOOP OVER EVERY 5 MIN PERIOD IN THE DAY
for houri in range(20,21):
    for mini in range(0,60,5):
        
        RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
        # add a string of format hh:mm:ss for printing
        RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] 
        
        print('working on ' + RadarFileTimePrint)
        # xgrid = xr.open_dataset('/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/' + \
        #                         RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc')


        NetCDFstoragePath = ('/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/'
                                                                 + RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc')
        # try to load in the netcdf file and if it doesn't work, just keep going through the loop
        try:
            xgrid = xr.open_dataset(NetCDFstoragePath)
        except FileNotFoundError:
            print(f'File missing for {RadarFileTimePrint}, skipping: {NetCDFstoragePath}')
            continue

        # index in the netcdf altitude variable for the altitude you want
        alti = np.where(xgrid.z == Altitude) # this is a double nested array for some reason
        alti = alti[0][0] # take the index out of the double nested array

        # quit out if the altitude does not correspond to one in the netCDF file
        if ( np.size(alti) != 1): 
            raise ValueError( str(Altitude) + ' m is not a valid altitude in the data')




        fig, ax = plt.subplots(figsize=(10, 8), 
                               subplot_kw={'projection': ccrs.PlateCarree()})

            # gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'
        # EXPERIMENTAL TOPO SHADING SECTION
        # EXPERIMENTAL TOPO SHADING SECTION
        # EXPERIMENTAL TOPO SHADING SECTION
        
        # TERRAIN SHADING USING LOCAL GEBCO DEM
        lon_min, lon_max = float(xgrid.lon.min()), float(xgrid.lon.max())
        lat_min, lat_max = float(xgrid.lat.min()), float(xgrid.lat.max())

        try:
            # Path to your GEBCO NetCDF
            gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'

            # Use the helper to get a subset over the radar domain
            dem_da = fetch_gebco_local(gebco_path, lon_min, lon_max, lat_min, lat_max)

            if dem_da is not None:
                # dem_da is an xarray.DataArray with coords lon, lat
                dem_lon = dem_da.lon.values
                dem_lat = dem_da.lat.values
                dem_data = dem_da.values

                # Make 2D lon/lat grids if necessary
                if dem_lon.ndim == 1 and dem_lat.ndim == 1:
                    dem_lon_2d, dem_lat_2d = np.meshgrid(dem_lon, dem_lat)
                else:
                    dem_lon_2d, dem_lat_2d = dem_lon, dem_lat

                # Ensure we have some valid data
                valid = np.isfinite(dem_data)
                if not np.any(valid):
                    raise ValueError('DEM has no finite values in this domain')

                colours = [
                    '#dde4e8',  # 0: pale blue-grey (ocean, < 0 m)
                
                    '#c4dec2',  # 1: 0–200 m, pale green
                    '#e4edc9',  # 2: 200–400 m, greenish-yellow
                    '#f3f0cf',  # 3: 400–600 m, pale yellow-beige
                    '#e9d7bd',  # 4: 600–800 m, light tan
                    '#ddc4aa',  # 5: 800–1000 m, tan
                    '#cfb194',  # 6: 1000–1200 m, light brown
                    '#b58f6e',  # 7: > 1200 m, darker brown
                ]
                
                bounds = [
                    -1000.0,  # ocean below 0
                    0.0,      # 0–200
                    200.0,    # 200–400
                    400.0,    # 400–600
                    600.0,    # 600–800
                    800.0,    # 800–1000
                    1000.0,   # 1000–1200
                    1200.0,   # > 1200
                    5000.0,
                ]
                
                cmap_elev = ListedColormap(colours)
                norm = BoundaryNorm(bounds, len(colours), clip=True)
                
                # Plot as semi‑transparent background
                elev_plot = ax.pcolormesh(
                    dem_lon_2d,
                    dem_lat_2d,
                    dem_data,
                    cmap=cmap_elev,
                    norm=norm,
                    alpha=1.0,
                    transform=ccrs.PlateCarree(),
                    # zorder=2,
                )

                # Draw 0 m contour as an accurate coastline
                coast_contour = ax.contour(
                    dem_lon_2d,
                    dem_lat_2d,
                    dem_data,
                    levels=[0.0],
                    colors='black',
                    linewidths=0.5,
                    transform=ccrs.PlateCarree(),
                    zorder=15,  # above radar and topo
                )

                # Draw 400 m contour
                coast_contour = ax.contour(
                    dem_lon_2d,
                    dem_lat_2d,
                    dem_data,
                    levels=[400.0],
                    colors='black',
                    linewidths=0.3,
                    transform=ccrs.PlateCarree(),
                    zorder=15,  # above radar and topo
                )

                print('Terrain shading (GEBCO, discrete bands) + coastline loaded successfully')

            else:
                raise ValueError('GEBCO DEM returned None')

        except Exception as e:
            print(f'Terrain shading failed: {e}')
            print('Falling back to simple land/ocean shading')
            ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.3, zorder=1)
            ax.add_feature(cfeature.LAND, facecolor='#E8E8E8', alpha=0.3, zorder=2)

        ax.add_feature(cfeature.BORDERS, linewidth=0.5, alpha=0.5, zorder=3)


        # EXPERIMENTAL TOPO SHADING SECTION
        # EXPERIMENTAL TOPO SHADING SECTION
        # EXPERIMENTAL TOPO SHADING SECTION

        GridViewer = ax.pcolormesh(xgrid.lon, xgrid.lat, xgrid[VarNameLong][0,alti,:,:], 
                                   cmap=VarColourBar , vmin=VarMinVal, vmax=VarMaxVal, transform=ccrs.PlateCarree())

        # Add MINOR gridlines (half degrees) - thin
        gl_minor = ax.gridlines(draw_labels=False, alpha=0.8, zorder=11, linewidth=0.3)
        gl_minor.xlocator = mticker.MultipleLocator(0.5)  # Every 0.5 degrees
        gl_minor.ylocator = mticker.MultipleLocator(0.5)  # Every 0.5 degrees

        # Add MAJOR gridlines (full degrees) - thick with labels
        gl_major = ax.gridlines(draw_labels=True, alpha=0.8, zorder=12, linewidth=0.5)
        gl_major.xlocator = mticker.MultipleLocator(1.0)  # Every 1 degree
        gl_major.ylocator = mticker.MultipleLocator(1.0)  # Every 1 degree
        
        # Format labels
        gl_major.xformatter = LONGITUDE_FORMATTER
        gl_major.yformatter = LATITUDE_FORMATTER
        
        # Remove labels from top and right
        gl_major.top_labels = False
        gl_major.right_labels = False
        gl_major.bottom_labels = True
        gl_major.left_labels = True

        # # Add coastlines ON TOP of radar data
        # ax.coastlines(resolution='10m', linewidth=0.5, color='black', zorder=13)
        
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')

        plt.title(VarName + ' for ' + RadarSiteName + ' Radar\nat ' + str(Altitude) + ' m Altitude\non ' + \
                  RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + ' at ' + \
                  str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] + ' UTC')
        plt.colorbar(GridViewer, ax=ax, label = VarName + ' [' + VarUnit + ']')

        SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/' + PlotType + '/' + RadarIDno + '/' + RadarFileDate + '/'
        SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + VarNameLong + '_' + \
                     PlotType + str(Altitude) + 'm.png'
        
        SavePath = SaveFolder + SaveFile
        
        if not Path(SaveFolder).exists():
            print('doing')
            Path(SaveFolder).mkdir(parents=True, exist_ok=True)
        
        # plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)
        # plt.close()
        

In [ ]:
xgrid.corrected_specific_differential_phase

In [ ]:
# GIF MAKER
# FOR Horizontal Cross Sections

PlotVar  = 'corrected_reflectivity'

SavedFolder = '/scratch/v46/sg3241/tmp/pngImages/Horz/' + RadarIDno + '/' + RadarFileDate + '/'
                
# LOADING IMAGES
files = sorted(os.listdir(SavedFolder)) # takes all of the files in the folder in the order they are named
images = [
    Image.open(os.path.join(SavedFolder, f))
    for f in files
    if f.endswith(PlotType + str(Altitude) + 'm.png')
]

# SaveFolder = '/cratch/v46/sg3241/tmp/pngImages/Vert/22/20240214/'

#         SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/CFAD/' + RadarIDno + '/' + RadarFileDate + '/'
#         SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + PlotVar + '_' + Version + 'CFAD.png'

GIFsaveFolder = '/scratch/v46/sg3241/tmp/gifImages/Horz/' + RadarIDno + '/' + RadarFileDate + '/'
GIFsaveFile   = RadarIDno + '_' + RadarFileDate + '_' + PlotVar + '_' + str(Altitude) + 'm.gif'

GIFsavePath = GIFsaveFolder + GIFsaveFile

if not Path(GIFsaveFolder).exists():
    Path(GIFsaveFolder).mkdir(parents=True, exist_ok=True)

# Save as looping GIF
images[0].save(
    GIFsavePath,
    save_all=True,
    append_images=images[1:],
    duration=200,    # ms per frame
    loop=0,          # 0 = loop forever
)
print('Saved GIF for ' + RadarFileDate)

In [ ]:
# THIS BLOCK RETRIEVES THE RELATIVE FREQUENCIES OF REFLECTIVITY VALUES TO PLOT IN THE CFAD

# what does the radar data use for the "fill value"?
FillValue = -32

MaxHeight = 10 # [km] maximum height you want to consider
ConsideredHeights = xgrid.z[np.where(xgrid.z<MaxHeight*1000)]

MinDBZ = -10 # [dBZ] minimum value of reflectivity you want to consider in the CFADs
MaxDBZ = 60 # [dBZ] minimum value of reflectivity you want to consider in the CFADs

NumHeights = np.size(ConsideredHeights) # the total number of altitudes in consideration

# find a floor and ceiling to the values of DBZ
LoEndDBZ = MinDBZ #int(np.floor(np.nanmin(flatZs)))
HiEndDBZ = MaxDBZ #int(np.ceil(np.nanmax(flatZs)))

NumDBZs = HiEndDBZ - LoEndDBZ # the total number of reflectivity [bins] in consideration

# create an empty array for storing frequencies and relative frequencies of reflectivity values
AbsCountsGrid = np.full([NumDBZs,NumHeights], np.nan)
NormCountsGrid = np.full([NumDBZs,NumHeights], np.nan)

# how many total grid cells in the radar data
TotalCells = np.size(xgrid.corrected_reflectivity)

# loop through each height and collect the relative frequencies
for zi in range(0, NumHeights):

    # store the flattened array of DBZ values
    flatZs = np.ndarray.flatten(np.array(xgrid.corrected_reflectivity[0,zi,:,:]))
    
    # replace the fill value with nans
    flatZs = np.where(flatZs == FillValue, np.nan, flatZs)
    
    # replace the low value with nans
    flatZs = np.where(flatZs < MinDBZ, np.nan, flatZs)
    
    # retrieve the data for a histogram
    
    counts, bins = np.histogram(flatZs[~np.isnan(flatZs)], bins=HiEndDBZ-LoEndDBZ, range=[LoEndDBZ, HiEndDBZ])

    # calculate the total cell frequencies and frequencies for that height
    AbsCounts  = counts * (1/TotalCells)
    NormCounts = counts * (1/np.sum(counts))
    
    # store away those counts and normalised counts in the grand CFAD data
    AbsCountsGrid[:,zi]  = AbsCounts
    NormCountsGrid[:,zi] = NormCounts

In [ ]:
# THIS BLOCK PLOTS A CFAD

Version = 'Absolute'

# based on the user's choice of version, pick the variable to plot, name the units its in, and set the colourbar limits for that variable
if (Version == 'Absolute'):
    PlotCounts = AbsCountsGrid * 100 # multiply by 100 to make it a percentage
    PlotUnit = '% of grid cells'
    MinVarVal = 0.00
    MaxVarVal = 0.10
elif (Version == 'Relative'):
    PlotCounts = NormCountsGrid  
    PlotUnit = 'per dBZ per km'
    MinVarVal = 0.00
    MaxVarVal = 0.25
else:
    sys.exit("please enter 'Absolute' or 'Relative' for Version")


# intervals of DBZ bins on the plot
DBZspacing = 1 # [dBZ]

DBZs = np.arange(LoEndDBZ,HiEndDBZ,DBZspacing) # (X COORDINATE ON THE PLOT) create a list of all possible DBZ values from -39 to 100 in steps of 1 
Heights = np.array(ConsideredHeights) * 0.001  # (Y COORDINATE ON THE PLOT) [converted to km] create a list of all of the altitudes where data are stored

# intervals of altitudes on the plot
HeightSpacing = 0.5 # [km] 

UprightFrequencies = np.transpose(PlotCounts) # (VALUES ON THE PLOT) transpose the stored relative frequencies of the reflectivities

# normalise frequencies to per unit DBZ per km of height
NormUprightFrequencies = UprightFrequencies * (1/DBZspacing) * (1/HeightSpacing)  

# PLOT A CFAD! (sort of)
fig, ax = plt.subplots(figsize=(8,6))

CFAD1 = pcolormeshC(DBZs , Heights, NormUprightFrequencies, ax=ax, cmap='nipy_spectral', vmin=MinVarVal, vmax=MaxVarVal)
plt.colorbar(CFAD1, ax=ax, label = Version + ' Frequency [' + PlotUnit + ']')

plt.xlim([MinDBZ, MaxDBZ])
plt.ylim([0,MaxHeight])

plt.title(Version + ' Frequencies of Corrected Reflectivity Values by Altitude\n for ' + RadarSiteName + ' Radar on ' + \
          RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + ' at ' + \
          str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] + ' UTC') 

ax.set_xlabel('Corrected Reflectivity [dBZ]')
ax.set_ylabel('Altitude [km]')

PlotVar  = 'corrected_reflectivity'

SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/CFAD/' + RadarIDno + '/' + RadarFileDate + '/'
SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + PlotVar + '_CFAD.png'

SavePath = SaveFolder + SaveFile

if not Path(SaveFolder).exists():
    print('Creating Folder: ' + SaveFolder)
    Path(SaveFolder).mkdir(parents=True, exist_ok=True)

plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)
plt.close()

In [ ]:
# CFAD LOOP
# ADDED 2026-06-10T18:56UTC+10:00

# what does the radar data use for the "fill value"?
FillValue = -32

# CHOOSE YOUR CFAD TYPE ('Absolute' or 'Relative')
Version = 'Absolute'

# CHOOSE YOUR RADAR
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Bris)

# name the radar site
if (RadarIDno == '22'):
    RadarSiteName = 'Mackay'
elif (RadarIDno == '106'):
    RadarSiteName = 'Townsville'
elif (RadarIDno == '66'):
    RadarSiteName = 'Mt Staplyton'
elif (RadarIDno == '50'):
    RadarSiteName = 'Marabong'
else:
    RadarSiteName = 'Site ' + RadarIDno


# CHOOSE YOUR DAY
# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 3
RadarDay   = 9

# CHOOSE YOUR ANNULUS OF CONSIDERATION
InnerRadius =  3  # [km] (inner radius of the annulus of consideration)
OuterRadius = 50  # [km] (outer radius of the annulus of consideration)

# CHOOSE YOUR HEIGHTS OF CONSIDERATION
MinHeight = 0.5 # [km] minimum height in the CFAD
MaxHeight = 15 # [km] maximum height in the CFAD

# CHOOSE YOUR HOUR and MINUTE
# houri = 00
# mini = 00

# LOOP OVER ALL SETS OF 5 MIN in the day
for houri in range(0,24):
    for mini in range(0,60,5):

        # add leading zeros for strings
        YYYY = str(RadarYear).zfill(4)
        MM = str(RadarMonth).zfill(2)
        DD = str(RadarDay).zfill(2)
        
        # write out the data in one string
        RadarFileDate  = YYYY + MM + DD 
        
        RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
        RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] 
        
        NetCDFstorageFolder = '/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/'
        NetCDFstorageFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc'
        NetCDFstoragePath = NetCDFstorageFolder + NetCDFstorageFile 
                                
        try:
            xgrid = xr.open_dataset(NetCDFstoragePath)
            print('working on reading the file for ' + RadarFileTimePrint)
        except FileNotFoundError:
            print(f'File missing for {RadarFileTimePrint}, skipping: {NetCDFstoragePath}')
            continue

        # calculate the distance from the radar and add it to the data frame as a new variable for data selection purposes
        xgrid['distance'] = distance = np.sqrt(xgrid['x']**2 + xgrid['y']**2)
        xgrid['distance'].attrs = {'long_name': 'Horizontal distance from radar', 'units': 'm'}
        
        ConsideredHeights = xgrid.z[ np.where( (xgrid.z>=MinHeight*1000) & (xgrid.z<=MaxHeight*1000) ) ]
        
        MinDBZ = -10 # [dBZ] minimum value of reflectivity you want to consider in the CFADs
        MaxDBZ = 60 # [dBZ] minimum value of reflectivity you want to consider in the CFADs
        
        NumHeights = np.size(ConsideredHeights) # the total number of altitudes in consideration
        
        # find a floor and ceiling to the values of DBZ
        LoEndDBZ = MinDBZ #int(np.floor(np.nanmin(flatZs)))
        HiEndDBZ = MaxDBZ #int(np.ceil(np.nanmax(flatZs)))
        
        NumDBZs = HiEndDBZ - LoEndDBZ # the total number of reflectivity [bins] in consideration

        # mask the reflectivities over only the specified annulus
        ConditionGridA = xgrid['distance'] > InnerRadius * 1000   # (y, x) boolean mask (converting from km to m)
        ConditionGridB = xgrid['distance'] < OuterRadius * 1000   # (y, x) boolean mask (converting from km to m)
        ConditionGridC = xgrid['z'] >= MinHeight * 1000   # (y, x) boolean mask (converting from km to m)
        ConditionGridD = xgrid['z'] <= MaxHeight * 1000   # (y, x) boolean mask (converting from km to m)
        
        ConditionGrid = ConditionGridA * ConditionGridB * ConditionGridC * ConditionGridD # combined boolean mask
        masked_reflectivity = xgrid['corrected_reflectivity'].where(ConditionGrid)  # MASK THE Z VALUES
        
        # create an empty array for storing frequencies and relative frequencies of reflectivity values
        AbsCountsGrid = np.full([NumDBZs,NumHeights], np.nan)
        NormCountsGrid = np.full([NumDBZs,NumHeights], np.nan)
        
        # how many total grid cells in the radar data
        # TotalCells = np.size(xgrid.corrected_reflectivity)
        TotalCells = float(np.sum(ConditionGrid)) # the total number of cells in consideration
        
        # loop through each height and collect the relative frequencies
        for zi in range(0, NumHeights):
        
            # store the flattened array of DBZ values
            flatZs = np.ndarray.flatten(np.array(masked_reflectivity[0,zi,:,:]))
            
            # replace the fill value with nans
            flatZs = np.where(flatZs == FillValue, np.nan, flatZs)
            
            # replace the low value with nans
            flatZs = np.where(flatZs < MinDBZ, np.nan, flatZs)

            
            # retrieve the data for a histogram
            counts, bins = np.histogram(flatZs[~np.isnan(flatZs)], bins=HiEndDBZ-LoEndDBZ, range=[LoEndDBZ, HiEndDBZ])
        
            # calculate the total cell frequencies and frequencies for that height
            AbsCounts  = counts * (1/TotalCells)
            NormCounts = counts * (1/np.sum(counts))
            
            # store away those counts and normalised counts in the grand CFAD data
            AbsCountsGrid[:,zi]  = AbsCounts
            NormCountsGrid[:,zi] = NormCounts
        
        
        # start plotting!
        
        # based on the user's choice of version, pick the variable to plot, name the units its in, and set the colourbar limits for that variable
        if (Version == 'Absolute'):
            PlotCounts = AbsCountsGrid * 100 # multiply by 100 to make it a percentage
            PlotUnit = '% of considered grid cells'
            MinVarVal = 0.00
            MaxVarVal = 0.50
        elif (Version == 'Relative'):
            PlotCounts = NormCountsGrid  
            PlotUnit = 'per dBZ per km'
            MinVarVal = 0.00
            MaxVarVal = 0.40
        else:
            sys.exit("please enter 'Absolute' or 'Relative' for Version")
        
        
        # intervals of DBZ bins on the plot
        DBZspacing = 1 # [dBZ]
        
        DBZs = np.arange(LoEndDBZ,HiEndDBZ,DBZspacing) # (X COORDINATE ON THE PLOT) create a list of all possible DBZ values from -39 to 100 in steps of 1 
        Heights = np.array(ConsideredHeights) * 0.001  # (Y COORDINATE ON THE PLOT) [converted to km] create a list of all of the altitudes where data are stored
        
        # intervals of altitudes on the plot
        HeightSpacing = 0.5 # [km] 
        
        UprightFrequencies = np.transpose(PlotCounts) # (VALUES ON THE PLOT) transpose the stored relative frequencies of the reflectivities
        
        # normalise frequencies to per unit DBZ per km of height
        NormUprightFrequencies = UprightFrequencies * (1/DBZspacing) * (1/HeightSpacing)  
        
        # PLOT A CFAD! (sort of)
        fig, ax = plt.subplots(figsize=(8,6))
        
        CFAD1 = pcolormeshC(DBZs , Heights, NormUprightFrequencies, ax=ax, cmap='nipy_spectral', vmin=MinVarVal, vmax=MaxVarVal)
        plt.colorbar(CFAD1, ax=ax, label = Version + ' Frequency [' + PlotUnit + ']')
        
        plt.xlim([MinDBZ, MaxDBZ])
        plt.ylim([0, MaxHeight])
        
        plt.title(Version + ' Frequencies of Corrected Reflectivity Values by Altitude\n for ' + RadarSiteName + ' Radar on ' + \
                  RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + ' at ' + \
                  str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] + ' UTC\n' + \
                 'For the Annulus between ' + str(InnerRadius) + ' and ' + str(OuterRadius) + ' km from the Radar') 
        
        ax.set_xlabel('Corrected Reflectivity [dBZ]')
        ax.set_ylabel('Altitude [km]')
        
        PlotVar  = 'corrected_reflectivity'
        
        SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/CFAD/' + RadarIDno + '/' + RadarFileDate + '/'
        SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + PlotVar + '_' + Version + 'CFAD.png'
        
        SavePath = SaveFolder + SaveFile
        
        if not Path(SaveFolder).exists():
            print('Creating Folder: ' + SaveFolder)
            Path(SaveFolder).mkdir(parents=True, exist_ok=True)
        
        plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)
        plt.close()

In [ ]:
def plot_2d(data, title='2D Plot', cmap='viridis'):
    """
    Plot a 2D array using pcolormesh.
    
    Parameters
    ----------
    data : array-like
        2D array to plot (e.g., 99x99)
    title : str
        Plot title
    cmap : str
        Colormap name (default: 'viridis')
    """
    fig, ax = plt.subplots(figsize=(8, 7))
    
    # Create pcolormesh plot
    pcm = ax.pcolormesh(data, cmap=cmap, shading='auto')
    
    # Add colorbar
    cbar = plt.colorbar(pcm, ax=ax)
    cbar.set_label('Value')
    
    # Labels and title
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_title(title)
    
    plt.tight_layout()
    plt.show()

# Usage
# plot_2d(reflectivity_masked[0].values, title='Reflectivity at z=0')


In [ ]:
plot_2d(ConditionGrid, title='Reflectivity at z=0')

In [ ]:
distance = np.sqrt(xgrid['x']**2 + xgrid['y']**2)

# Add as a new variable to the dataset
xgrid['distance'] = distance
xgrid['distance'].attrs = {
    'long_name': 'Horizontal distance from radar',
    'units': 'm'
}

In [ ]:
xgrid['distance'][150][100]

In [ ]:
# GIF MAKER
# FOR CFADS

# # CHOOSE YOUR DAY
# # the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
# RadarYear  = 2024
# RadarMonth = 3
# RadarDay   = 9

# # add leading zeros for strings
# YYYY = str(RadarYear).zfill(4)
# MM = str(RadarMonth).zfill(2)
# DD = str(RadarDay).zfill(2)

# # # write out the data in one string
# # RadarFileDate  = YYYY + MM + DD 

# Version = 'Relative'
# PlotVar  = 'corrected_reflectivity'

SavedFolder = '/scratch/v46/sg3241/tmp/pngImages/CFAD/' + RadarIDno + '/' + RadarFileDate + '/'
                
# LOADING IMAGES
files = sorted(os.listdir(SavedFolder)) # takes all of the files in the folder in the order they are named
images = [
    Image.open(os.path.join(SavedFolder, f))
    for f in files
    if f.endswith(Version + 'CFAD.png')
]

# SaveFolder = '/cratch/v46/sg3241/tmp/pngImages/Vert/22/20240214/'

#         SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/CFAD/' + RadarIDno + '/' + RadarFileDate + '/'
#         SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + PlotVar + '_' + Version + 'CFAD.png'

GIFsaveFolder = '/scratch/v46/sg3241/tmp/gifImages/CFAD/' + RadarIDno + '/' + RadarFileDate + '/'
GIFsaveFile   = RadarIDno + '_' + RadarFileDate + '_' + PlotVar + '_' + Version + 'CFAD.gif'

GIFsavePath = GIFsaveFolder + GIFsaveFile

if not Path(GIFsaveFolder).exists():
    Path(GIFsaveFolder).mkdir(parents=True, exist_ok=True)

# Save as looping GIF
images[0].save(
    GIFsavePath,
    save_all=True,
    append_images=images[1:],
    duration=200,    # ms per frame
    loop=0,          # 0 = loop forever
)
print('Saved GIF for ' + RadarFileDate)

In [ ]:
# TRY AND MAKE A STITCHED GIF


from PIL import Image

gif_files = [
    "/scratch/v46/sg3241/tmp/gifImages/Horz/22/20240214/22_20240214_corrected_reflectivity_2000m.gif",
    "/scratch/v46/sg3241/tmp/gifImages/Vert/22/20240214/22_20240214_corrected_reflectivity_40South.gif",
    "/scratch/v46/sg3241/tmp/gifImages/CFAD/22/20240214/22_20240214_corrected_reflectivity_RelativeCFAD.gif",
    "/scratch/v46/sg3241/tmp/gifImages/CFAD/22/20240214/22_20240214_corrected_reflectivity_AbsoluteCFAD.gif",
]

# Open GIFs
gifs = [Image.open(f) for f in gif_files]

# Check frame counts
n_frames = gifs[0].n_frames
for gif in gifs:
    if gif.n_frames != n_frames:
        raise ValueError("All GIFs must have the same number of frames")

frames = []

for frame_idx in range(n_frames):
    imgs = []

    for gif in gifs:
        gif.seek(frame_idx)
        imgs.append(gif.convert("RGBA"))

    w, h = imgs[0].size

    # Create 2x2 canvas
    combined = Image.new("RGBA", (2 * w, 2 * h))

    combined.paste(imgs[0], (0, 0))
    combined.paste(imgs[1], (w, 0))
    combined.paste(imgs[2], (0, h))
    combined.paste(imgs[3], (w, h))

    frames.append(combined)

# Use duration from first GIF
gifs[0].seek(0)
duration = gifs[0].info.get("duration", 100)

frames[0].save(
    "/scratch/v46/sg3241/tmp/gifImages/Stitched/22_20240214_corrected_reflectivity.gif",
    save_all=True,
    append_images=frames[1:],
    duration=duration,
    loop=0,
    disposal=2,
)

In [ ]:
# TRY AND MAKE A STITCHED GIF


from PIL import Image

gif_files = [
    "/scratch/v46/sg3241/tmp/gifImages/Horz/22/20240309/22_20240309_corrected_reflectivity_2000m.gif",
    "/scratch/v46/sg3241/tmp/gifImages/Vert/22/20240309/22_20240309_corrected_reflectivity_20North.gif",
    "/scratch/v46/sg3241/tmp/gifImages/CFAD/22/20240309/22_20240309_corrected_reflectivity_RelativeCFAD.gif",
    "/scratch/v46/sg3241/tmp/gifImages/CFAD/22/20240309/22_20240309_corrected_reflectivity_AbsoluteCFAD.gif",
]

# Open GIFs
gifs = [Image.open(f) for f in gif_files]

# Check frame counts
n_frames = gifs[0].n_frames
for gif in gifs:
    if gif.n_frames != n_frames:
        raise ValueError("All GIFs must have the same number of frames")

frames = []

for frame_idx in range(n_frames):
    imgs = []

    for gif in gifs:
        gif.seek(frame_idx)
        imgs.append(gif.convert("RGBA"))

    w, h = imgs[0].size

    # Create 2x2 canvas
    combined = Image.new("RGBA", (2 * w, 2 * h))

    combined.paste(imgs[0], (0, 0))
    combined.paste(imgs[1], (w, 0))
    combined.paste(imgs[2], (0, h))
    combined.paste(imgs[3], (w, h))

    frames.append(combined)

# Use duration from first GIF
gifs[0].seek(0)
duration = gifs[0].info.get("duration", 100)

frames[0].save(
    "/scratch/v46/sg3241/tmp/gifImages/Stitched/22_20240309_corrected_reflectivity.gif",
    save_all=True,
    append_images=frames[1:],
    duration=duration,
    loop=0,
    disposal=2,
)